### np.fft.ifft

In [1]:
import math
import cmath

def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def _is_power_of_2(n):
    return n > 0 and (n & (n - 1)) == 0


def _dft_naive(x):
    N = len(x)
    out = []
    for n in range(N):
        val = 0 + 0j
        for k in range(N):
            angle = 2 * math.pi * k * n / N
            val += x[k] * cmath.exp(1j * angle)
        out.append(val)
    return out


def np_fft_ifft(a, n=None, axis=-1, norm=None):
    """
    Pure-Python equivalent of np.fft.ifft for 1-D input.

    - a: 1-D list of complex/real numbers in frequency domain
    - n: optional output length (truncate or zero-pad input)
    - axis: only axis=-1 / axis=0 for 1-D supported
    - norm: None/'backward', 'forward', or 'ortho'
    """
    shape = get_shape(a)
    if len(shape) == 0:
        raise ValueError("Input must be at least 1-D")
    if len(shape) != 1:
        raise NotImplementedError("Only 1-D input supported in this version")

    if axis not in (-1, 0):
        raise ValueError(f"axis {axis} is out of bounds for array of dimension 1")

    x = [complex(v) for v in a]

    if n is not None:
        if not isinstance(n, int) or n <= 0:
            raise ValueError("n must be a positive integer")
        if n < len(x):
            x = x[:n]
        elif n > len(x):
            x = x + [0j] * (n - len(x))

    N = len(x)
    if N == 0:
        return []

    # inverse DFT directly
    result = _dft_naive(x)

    # divide by N unless norm says otherwise
    if norm is None or norm == "backward":
        result = [v / N for v in result]
    elif norm == "forward":
        pass
    elif norm == "ortho":
        factor = math.sqrt(N)
        result = [v / factor for v in result]
    else:
        raise ValueError(f"Unknown norm mode: '{norm}'")

    return result

### test cases

In [2]:
print(np_fft_ifft([1, 1, 1, 1]))
print(np_fft_ifft([4, 0, 0, 0]))
print(np_fft_ifft([0, 4, 0, 0]))
print(np_fft_ifft([0, 0, 4, 0]))
print(np_fft_ifft([0, 0, 0, 4]))
print(np_fft_ifft([1+0j, 0+0j, 0+0j, 0+0j]))
print(np_fft_ifft([1, 0, 0, 0, 0, 0, 0, 0], norm="ortho"))
print(np_fft_ifft([1, 0, 0, 0], n=8))
print(np_fft_ifft([4, 0, 0, 0], norm="forward")) 

[(1+0j), (-4.592425496802574e-17+5.551115123125783e-17j), 6.123233995736765e-17j, (8.226161367281941e-17+8.326672684688674e-17j)]
[(1+0j), (1+0j), (1+0j), (1+0j)]
[(1+0j), (6.123233995736766e-17+1j), (-1+1.2246467991473532e-16j), (-1.8369701987210297e-16-1j)]
[(1+0j), (-1+1.2246467991473532e-16j), (1-2.4492935982947064e-16j), (-1+3.6739403974420594e-16j)]
[(1+0j), (-1.8369701987210297e-16-1j), (-1+3.6739403974420594e-16j), (5.51091059616309e-16+1j)]
[(0.25+0j), (0.25+0j), (0.25+0j), (0.25+0j)]
[(0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j)]
[(0.125+0j), (0.125+0j), (0.125+0j), (0.125+0j), (0.125+0j), (0.125+0j), (0.125+0j), (0.125+0j)]
[(4+0j), (4+0j), (4+0j), (4+0j)]


In [3]:
print(np_fft_ifft(42)) 

ValueError: Input must be at least 1-D

In [4]:
print(np_fft_ifft([1, 2, 3], norm="invalid")) 

ValueError: Unknown norm mode: 'invalid'

In [5]:
print(np_fft_ifft([1, 2, 3], n=-5)) 

ValueError: n must be a positive integer

In [6]:
print(np_fft_ifft([1, 2, 3], axis=1)) 

ValueError: axis 1 is out of bounds for array of dimension 1

In [7]:
print(np_fft_ifft([[1, 2], [3]])) 

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)